# Test of S1-ARD processor

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-766

In [1]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *

init_demo()
init_dask_cluster_eopf(scale=4, processor_name="s1ard", processor_code="S1ARD")

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  

# You can check here the number of workers, threads and memory per worker.
# In local mode, you can configure them by running e.g.
# DASK_MEMORY_EOPF=4G DASK_THREADS_EOPF=4 docker compose up # ...
display(dask_cluster_eopf)

Auxip service: http://rs-server-adgs:8000/auxip
CADIP service: http://rs-server-cadip:8000/cadip
Catalog service: http://rs-server-catalog:8000
Staging service: http://rs-server-staging:8000
DPR service: http://rs-dpr-service:8000
Connecting to dask gateway for 'dask-s1ard': http://dask-s1-ard:8000 ...
Create new dask cluster
Dask dashboard for 'dask-s1ard': http://localhost:8704/clusters/84ab94c6be5e4514ab465b5775b75fca/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1394: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| tornado | 6.3.3  | 6.4.2     | None    |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Dask workers for 'dask-s1ard' are up: 0/4
Dask workers for 'dask-s1ard' are up: 4/4


In [2]:
# Other imports
import glob
import shutil
import time
import os.path as osp

from datetime import timedelta
from IPython.display import JSON

from rs_common.prefect_utils import *

## Setup for S1-ARD processor

In [3]:
# Get the prefect share bucket folder
share_bucket = await get_share_bucket()

# s3 bucket dirs that will contain the data
s3_base = osp.join(
    "s3://",
    share_bucket.bucket_name,
    share_bucket.bucket_folder,
    "users",
    OWNER_ID,
    "l0",
)
s3_config_dir = osp.join(s3_base, "config")
s3_output_dir = osp.join(s3_base, "output")
s3_report_dir = osp.join(s3_base, "reports")

# Data properties
s1_ard = {
    "process_name": "s1_ard", # process name in rs-dpr-service       # TODO: n'existe pas dans rs-dpr-service
    "payload_subpath": "s1-ard/demo_joborder.yaml", # payload file path relative to the config dir
    "s3_output_dir": f"{s3_output_dir}/s1_ard", # output dir in the s3 bucket
    "s3_report_dir": f"{s3_report_dir}/s1_ard", # report dir in the s3 bucket
}

# Upload the local configuration dir to s3 bucket
await s3_upload_dir("./config", s3_config_dir)

# Update local secret file depending on the environment, 
# and upload it again to the s3 bucket.
await dpr_client.update_configuration(
    local_path = "./config/secrets.json",
    s3_path = osp.join(s3_config_dir, "secrets.json"),
)

15:42:09.956 | INFO    | prefect.S3Bucket - Uploading from 'config/secrets.json' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/ecombelles/l0/config/secrets.json'.

15:42:09.960 | INFO    | prefect.S3Bucket - Uploading from 'config/logging_config.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/ecombelles/l0/config/logging_config.yaml'.

15:42:09.963 | INFO    | prefect.S3Bucket - Uploading from 'config/s1-ard/ard.json' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/ecombelles/l0/config/s1-ard/ard.json'.

15:42:09.965 | INFO    | prefect.S3Bucket - Uploading from 'config/s1-ard/demo_joborder.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/ecombelles/l0/config/s1-ard/demo_joborder.yaml'.

15:42:09.968 | INFO    | prefect.S3Bucket - Uploading from 'config/s1-ard/.ipynb_checkpoints/ard-checkpoint.json' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/ecombelles/l0/config/s1-ard/.ipynb_checkpoints/ard-checkpoint.json'.

15:42:09.970 | INFO    | prefect.S3Bucket - Uploading from 'config/s1-ard/.ipynb_checkpoints/demo_joborder-checkpoint.yaml' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/ecombelles/l0/config/s1-ard/.ipynb_checkpoints/demo_joborder-checkpoint.yaml'.

15:42:10.054 | INFO    | prefect.S3Bucket - Uploaded 6 files from 'config' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/ecombelles/l0/config/s1-ard/.ipynb_checkpoints/demo_joborder-checkpoint.yaml'

15:42:10.073 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpqr34rioa' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/ecombelles/l0/config/secrets.json'.

'prefect-share/users/ecombelles/l0/config/secrets.json'

In [4]:
# Number of dask workers
N_WORKERS = len(dask_client_eopf.scheduler_info()["workers"])

async def process_data(data: dict):
    """Process s1_ard data"""

    # Remove existing output and report folders
    s3_delete(data["s3_output_dir"], log=True)
    s3_delete(data["s3_report_dir"], log=True)
    
    # Update local payload file depending on the environment, upload it to the s3 bucket,
    # and initialize output bucket folders.
    await dpr_client.update_configuration(
        local_path = osp.join("./config", data["payload_subpath"]),
        s3_path = osp.join(s3_config_dir, data["payload_subpath"]),
        is_payload = True,
        # Specific environment variables to expand in the payload file
        N_WORKERS=N_WORKERS,
        PREFECT_BUCKET_NAME=os.environ["PREFECT_BUCKET_NAME"], 
        OUTPUT_DIR=data["s3_output_dir"],
    )
    
    # Run processor
    start_time = time.time()
    result = dpr_client.run_process(
        data["process_name"],
        s3_config_dir = s3_config_dir,
        payload_subpath = data["payload_subpath"],
        s3_report_dir = data["s3_report_dir"],
    )
    try:
        dpr_client.wait_for_job(result, logger=logger, poll_interval=5)
    finally:
        print(f"Processor execution time: {str(timedelta(seconds=time.time() - start_time))}")
    
        # Download reports folder from the s3 bucket
        local_report_dir = f"./reports/{Path(data['s3_report_dir']).name}"
        shutil.rmtree(local_report_dir, ignore_errors=True)
        await s3_download_dir(data["s3_report_dir"], local_report_dir)
        
        # Display logs here
        local_log_file = glob.glob(osp.join(local_report_dir, "**/*.processor.log"), recursive=True)[0]
        with open(local_log_file, "r", encoding="utf-8") as openend:
            print(f"Log file {local_log_file!r}:\n{openend.read()}")

    # Download output data from the s3 bucket
    print(f"Output product generated on: {data['s3_output_dir']!r}")
    local_output_dir = f"./outputs/{Path(data['s3_output_dir']).name}"
    await s3_download_dir(data["s3_output_dir"], local_output_dir)
    print(f"Open it locally with QGis from: {osp.realpath(local_output_dir)!r}")

## Run processor

In [5]:
# Run S1-ARD
await process_data(s1_ard)

15:42:16.167 | INFO    | prefect.S3Bucket - Delete from 'http://minio:9000': [
  "s3://rs-dev-cluster-temp/prefect-share/users/ecombelles/l0/reports/s1_ard/demo_joborder.processor.log",
  "s3://rs-dev-cluster-temp/prefect-share/users/ecombelles/l0/reports/s1_ard/logging_config.log"
]

15:42:16.195 [INFO] (rs_client.rs_client) Write empty file: 20 's3://rs-dev-cluster-temp/ARD_V2/OUTPUT_DATA/S01SIWCSL_20240416T171518_0027_A015_S000_00000_DV.zarr/.empty'


15:42:16.226 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmp1kgcki2k' to the bucket 'rs-dev-cluster-temp' path 'ARD_V2/OUTPUT_DATA/S01SIWCSL_20240416T171518_0027_A015_S000_00000_DV.zarr/.empty'.

15:42:16.230 [INFO] (rs_client.rs_client) Write empty file: 20 's3://rs-dev-cluster-temp/ARD_V2/OUTPUT_DATA/S01SIWNRB_20240416T171518_0027_A015_S000_00000_DV.zarr/.empty'


15:42:16.253 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmprmkilboz' to the bucket 'rs-dev-cluster-temp' path 'ARD_V2/OUTPUT_DATA/S01SIWNRB_20240416T171518_0027_A015_S000_00000_DV.zarr/.empty'.

15:42:16.285 | INFO    | prefect.S3Bucket - Uploaded from '/tmp/tmpitqst7du' to the bucket 'rs-dev-cluster-temp' path 'prefect-share/users/ecombelles/l0/config/s1-ard/demo_joborder.yaml'.

15:42:16.684 [INFO] (resources.utils) job_status: {'progress': 0, 'created': '2025-08-28T15:42:16Z', 'message': 'Sending task to the dask cluster', 'status': 'running', 'processID': 'dpr-service', 'type': 'process', 'started': '2025-08-28T15:42:16Z', 'updated': '2025-08-28T15:42:16Z', 'jobID': 'dcfc4f90-2dd2-4200-bde2-9ab2969dafb5'}
15:42:16.686 [INFO] (resources.utils) -----  job 'dcfc4f90-2dd2-4200-bde2-9ab2969dafb5': RUNNING 

15:42:21.717 [INFO] (resources.utils) job_status: {'progress': 50, 'created': '2025-08-28T15:42:16Z', 'message': 'In progress', 'status': 'running', 'processID': 'dpr-service', 'type': 'process', 'started': '2025-08-28T15:42:16Z', 'updated': '2025-08-28T15:42:16Z', 'jobID': 'dcfc4f90-2dd2-4200-bde2-9ab2969dafb5'}
15:42:21.719 [INFO] (resources.utils) -----  job 'dcfc4f90-2dd2-4200-bde2-9ab2969dafb5': RUNNING 

15:42:26.756 [INFO] (resources.utils) job_status: {'progress': 50, 'created': '2025-08-28T15:42:16Z', 'message': 'In progress', 'status': 'running', 'pr

Processor execution time: 0:00:15.521371
Log file './reports/s1_ard/demo_joborder.processor.log':
INFO:eopf.trigger.local:RUN with {'variables': {'cslcs_path': 's3://rs-dev-cluster-temp/ARD_V2/OUTPUT_DATA/S01SIWCSL_20240416T171518_0027_A015_S000_00000_DV.zarr', 'nrb_path': 's3://rs-dev-cluster-temp/ARD_V2/OUTPUT_DATA/S01SIWNRB_20240416T171518_0027_A015_S000_00000_DV.zarr', 'reference_date': '2024-04-16', 'dynamic_working_directory': 's3://rs-dev-cluster-temp/ARD_V2/WDIR_DYNAMIC', 'static_working_directory': 's3://rs-dev-cluster-temp/ARD_V2/WDIR_STATIC'}, 'general_configuration': {'logging': {'level': 'DEBUG'}, 'triggering__use_basic_logging': True, 'triggering__validate_run': False}, 'workflow': [{'name': 'calibration', 'module': 's1_l12_rp.computing.ard_processing_units', 'processing_unit': 'Calibration', 'adfs': {'CONFIG': 'CONFIG', 'ETAD': 'ETAD'}, 'parameters': {'slcs_path': ['s3://rs-dev-cluster-temp/ARD_V2/SAFE/S1A_IW_SLC__1SDV_20240416T171518_20240416T171545_053462_067C88_DA9D.S

RuntimeError:  job 'dcfc4f90-2dd2-4200-bde2-9ab2969dafb5': FAILED

## Shutdown cluster

In [ ]:
shutdown = False
if shutdown:    
    # You can scale the clusters to 0 workers
    dask_gateway_eopf.scale_cluster(dask_cluster_eopf.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway_eopf, dask_cluster_eopf.name)

    # Close the python objects
    close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.